# Qwen3 TTS 12Hz 0.6B Base - Colab Load Check

This notebook loads `Qwen/Qwen3-TTS-12Hz-0.6B-Base`, provides a simple voice-clone TTS widget, and benchmarks latency and generated-audio throughput on a Colab T4 GPU.

Use `Runtime -> Change runtime type -> T4 GPU` before running the model cells.

In [ ]:
# Install runtime dependencies. Restart the runtime if Colab asks for it.
!pip install -U -q qwen-tts ipywidgets soundfile librosa pandas

In [ ]:
import base64
import gc
import json
import math
import os
import statistics
import subprocess
import time
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, Javascript, clear_output, display
from google.colab import output
from ipywidgets import Button, Checkbox, Dropdown, FileUpload, HBox, IntSlider, Output, Text, Textarea, VBox
from qwen_tts import Qwen3TTSModel

MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-0.6B-Base'
WORK_DIR = Path('/content/qwen3-tts-poc')
WORK_DIR.mkdir(parents=True, exist_ok=True)

def choose_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

MODEL_DTYPE = choose_dtype()
DEVICE_MAP = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print({'model_id': MODEL_ID, 'device_map': DEVICE_MAP, 'dtype': str(MODEL_DTYPE)})

In [ ]:
def gpu_snapshot():
    if not torch.cuda.is_available():
        return {'gpu': None, 'allocated_mb': None, 'reserved_mb': None, 'max_allocated_mb': None}
    return {
        'gpu': torch.cuda.get_device_name(0),
        'allocated_mb': round(torch.cuda.memory_allocated() / 1024**2, 1),
        'reserved_mb': round(torch.cuda.memory_reserved() / 1024**2, 1),
        'max_allocated_mb': round(torch.cuda.max_memory_allocated() / 1024**2, 1),
    }

def reset_gpu_peak():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

print(gpu_snapshot())
!nvidia-smi

In [ ]:
reset_gpu_peak()
load_started = time.perf_counter()
tts_model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map=DEVICE_MAP,
    dtype=MODEL_DTYPE,
)
load_seconds = time.perf_counter() - load_started
print({'load_seconds': round(load_seconds, 3), 'gpu': gpu_snapshot()})

In [ ]:
RECORD_JS = """
async function recordAudio(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const mediaRecorder = new MediaRecorder(stream);
  const chunks = [];
  mediaRecorder.ondataavailable = event => chunks.push(event.data);
  mediaRecorder.start();
  await new Promise(resolve => setTimeout(resolve, seconds * 1000));
  await new Promise(resolve => {
    mediaRecorder.onstop = resolve;
    mediaRecorder.stop();
  });
  stream.getTracks().forEach(track => track.stop());
  const blob = new Blob(chunks, {type: 'audio/webm'});
  const reader = new FileReader();
  const dataUrl = await new Promise(resolve => {
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  });
  return dataUrl;
}
"""

def convert_to_wav(audio_path, sample_rate=24000):
    audio_path = Path(audio_path)
    if audio_path.suffix.lower() == '.wav':
        return audio_path
    wav_path = audio_path.with_suffix('.wav')
    subprocess.run(['ffmpeg', '-y', '-i', str(audio_path), '-ac', '1', '-ar', str(sample_rate), str(wav_path)], check=True, capture_output=True)
    return wav_path

def record_audio(seconds=5, out_path=WORK_DIR / 'reference.webm'):
    display(Javascript(RECORD_JS))
    data_url = output.eval_js(f'recordAudio({float(seconds)})')
    payload = data_url.split(',', 1)[1]
    out_path = Path(out_path)
    out_path.write_bytes(base64.b64decode(payload))
    return convert_to_wav(out_path)

def save_uploaded_audio(upload_value, out_path=WORK_DIR / 'reference_audio'):
    if not upload_value:
        raise ValueError('Upload a reference audio file first.')
    item = next(iter(upload_value.values())) if isinstance(upload_value, dict) else upload_value[0]
    name = item.get('metadata', {}).get('name') or item.get('name') or 'reference_audio'
    suffix = Path(name).suffix or '.wav'
    out_file = Path(str(out_path) + suffix)
    out_file.write_bytes(item['content'])
    return convert_to_wav(out_file)

def audio_duration_seconds_from_array(wav, sr):
    return float(len(wav) / sr)

def generate_tts(text, language, ref_audio, ref_text, max_new_tokens=2048, reuse_prompt=True):
    reset_gpu_peak()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    before = gpu_snapshot()
    started = time.perf_counter()
    prompt = None
    if reuse_prompt:
        prompt = tts_model.create_voice_clone_prompt(ref_audio=str(ref_audio), ref_text=ref_text)
        prompt_ready_seconds = time.perf_counter() - started
        wavs, sr = tts_model.generate_voice_clone(text=text, language=language, voice_clone_prompt=prompt, max_new_tokens=max_new_tokens)
    else:
        prompt_ready_seconds = None
        wavs, sr = tts_model.generate_voice_clone(text=text, language=language, ref_audio=str(ref_audio), ref_text=ref_text, max_new_tokens=max_new_tokens)
    latency = time.perf_counter() - started
    wav = np.asarray(wavs[0])
    out_path = WORK_DIR / f'tts_{int(time.time() * 1000)}.wav'
    sf.write(out_path, wav, sr)
    output_seconds = audio_duration_seconds_from_array(wav, sr)
    return {
        'output_path': str(out_path),
        'output_seconds': output_seconds,
        'latency_seconds': latency,
        'audio_seconds_per_runtime_minute': output_seconds / (latency / 60) if latency else None,
        'prompt_ready_seconds': prompt_ready_seconds,
        'ttfb_seconds': None,
        'gpu_before': before,
        'gpu_after': gpu_snapshot(),
    }

In [ ]:
text = Textarea(value='This is a short Qwen3 text to speech latency check on a Colab T4 GPU.', description='Text', layout={'width': '900px', 'height': '90px'})
language = Dropdown(options=['English', 'Spanish', 'French', 'German', 'Italian', 'Portuguese', 'Chinese', 'Japanese', 'Korean', 'Russian', 'Auto'], value='English', description='Language')
ref_text = Textarea(value='This is my short reference voice sample for cloning.', description='Ref text', layout={'width': '900px', 'height': '70px'})
record_seconds = IntSlider(value=5, min=3, max=20, step=1, description='Ref sec')
max_tokens = IntSlider(value=2048, min=256, max=4096, step=256, description='Max tokens')
reuse_prompt = Checkbox(value=True, description='Precompute prompt')
upload = FileUpload(accept='audio/*', multiple=False, description='Upload ref')
record_button = Button(description='Record reference', button_style='')
upload_button = Button(description='Use upload', button_style='')
generate_button = Button(description='Generate speech', button_style='primary')
out = Output()
state = {'ref_audio': None}

def on_record(_):
    with out:
        clear_output()
        state['ref_audio'] = record_audio(record_seconds.value)
        print(f'Reference audio: {state["ref_audio"]}')
        display(Audio(str(state['ref_audio'])))

def on_upload(_):
    with out:
        clear_output()
        state['ref_audio'] = save_uploaded_audio(upload.value)
        print(f'Reference audio: {state["ref_audio"]}')
        display(Audio(str(state['ref_audio'])))

def on_generate(_):
    with out:
        clear_output()
        if state['ref_audio'] is None:
            raise ValueError('Record or upload reference audio first.')
        result = generate_tts(text.value, language.value, state['ref_audio'], ref_text.value, max_new_tokens=max_tokens.value, reuse_prompt=reuse_prompt.value)
        print(json.dumps({k: v for k, v in result.items() if k not in ['gpu_before', 'gpu_after']}, indent=2))
        display(Audio(result['output_path']))

record_button.on_click(on_record)
upload_button.on_click(on_upload)
generate_button.on_click(on_generate)
display(VBox([text, HBox([language, max_tokens, reuse_prompt]), ref_text, HBox([record_seconds, record_button, upload, upload_button, generate_button]), out]))

In [ ]:
def percentile(values, p):
    values = sorted(float(v) for v in values if v is not None and not math.isnan(float(v)))
    if not values:
        return None
    idx = (len(values) - 1) * p / 100
    lo = math.floor(idx)
    hi = math.ceil(idx)
    if lo == hi:
        return values[int(idx)]
    return values[lo] + (values[hi] - values[lo]) * (idx - lo)

def summarize_tts_runs(rows):
    successes = [r for r in rows if r['ok']]
    latencies = [r['latency_seconds'] for r in successes]
    total_output_seconds = sum(r['output_seconds'] for r in successes)
    total_runtime_seconds = sum(r['latency_seconds'] for r in successes)
    throughput = (total_output_seconds / 60) / (total_runtime_seconds / 60) if total_runtime_seconds else None
    return {
        'runs': len(rows),
        'successes': len(successes),
        'failures': len(rows) - len(successes),
        'audio_minutes_processed_per_runtime_minute': throughput,
        'mean_latency_seconds': statistics.mean(latencies) if latencies else None,
        'p50_latency_seconds': percentile(latencies, 50),
        'p90_latency_seconds': percentile(latencies, 90),
        'p95_latency_seconds': percentile(latencies, 95),
        'mean_output_seconds': statistics.mean([r['output_seconds'] for r in successes]) if successes else None,
        'ttfb_seconds': 'N/A - offline package call returns only the final result',
        'gpu_memory_after': gpu_snapshot(),
    }

def run_tts_load_test(prompts, ref_audio, ref_text, language='English', repeats=5, warmup_runs=1, batch_sizes=None, max_new_tokens=2048, reuse_prompt=True):
    batch_sizes = batch_sizes or [1]
    rows = []
    prompt_cache = None
    if reuse_prompt:
        prompt_cache = tts_model.create_voice_clone_prompt(ref_audio=str(ref_audio), ref_text=ref_text)
    for batch_size in batch_sizes:
        for i in range(warmup_runs + repeats):
            is_warmup = i < warmup_runs
            prompt_batch = [prompts[(i + j) % len(prompts)] for j in range(int(batch_size))]
            language_batch = [language] * int(batch_size)
            try:
                started = time.perf_counter()
                if prompt_cache is not None:
                    wavs, sr = tts_model.generate_voice_clone(text=prompt_batch, language=language_batch, voice_clone_prompt=prompt_cache, max_new_tokens=max_new_tokens)
                else:
                    wavs, sr = tts_model.generate_voice_clone(text=prompt_batch, language=language_batch, ref_audio=str(ref_audio), ref_text=ref_text, max_new_tokens=max_new_tokens)
                latency = time.perf_counter() - started
                output_seconds = sum(audio_duration_seconds_from_array(np.asarray(wav), sr) for wav in wavs)
                rows.append({'batch_size': int(batch_size), 'run': i + 1, 'warmup': is_warmup, 'ok': True, 'output_seconds': output_seconds, 'latency_seconds': latency, 'ttfb_seconds': None, 'error': None})
            except Exception as exc:
                rows.append({'batch_size': int(batch_size), 'run': i + 1, 'warmup': is_warmup, 'ok': False, 'output_seconds': None, 'latency_seconds': None, 'ttfb_seconds': None, 'error': repr(exc)})
    measured = [r for r in rows if not r['warmup']]
    return pd.DataFrame(rows), summarize_tts_runs(measured)

In [ ]:
# Benchmark using the reference audio selected in the widget above, or set REF_AUDIO_PATH manually.
REF_AUDIO_PATH = state.get('ref_audio')
REF_TEXT = ref_text.value
PROMPTS = [
    'This is a short latency benchmark sentence for text to speech.',
    'The second sample checks repeated generation throughput on the same GPU.',
    'A longer prompt helps estimate how performance changes with more generated speech.',
]
REPEATS = 5
WARMUP_RUNS = 1
BATCH_SIZES = [1]

if REF_AUDIO_PATH is None or not Path(REF_AUDIO_PATH).exists():
    raise FileNotFoundError('Record/upload reference audio first, or set REF_AUDIO_PATH to an existing audio file.')

tts_rows, tts_summary = run_tts_load_test(PROMPTS, REF_AUDIO_PATH, REF_TEXT, language=language.value, repeats=REPEATS, warmup_runs=WARMUP_RUNS, batch_sizes=BATCH_SIZES, max_new_tokens=max_tokens.value, reuse_prompt=True)
display(tts_rows)
display(pd.DataFrame([tts_summary]))
tts_rows.to_csv(WORK_DIR / 'tts_load_test_rows.csv', index=False)
pd.DataFrame([tts_summary]).to_csv(WORK_DIR / 'tts_load_test_summary.csv', index=False)
print('Cost formula: eur_per_audio_min = eur_per_runtime_min / audio_minutes_processed_per_runtime_minute')